# Import Dataset Hourly V2

Notebook ini mengambil data **hourly saja** dari Open-Meteo Archive API dengan fitur per jam yang diperluas (tanpa data daily).

### Hourly Features (11 total)
- `temperature_2m`, `weather_code`, `relative_humidity_2m`, `pressure_msl`, `wind_speed_10m`
- `rain`, `apparent_temperature`, `surface_pressure`, `dew_point_2m`, `wind_direction_10m`, `wind_gusts_10m`


In [11]:
import requests
import pandas as pd
from datetime import datetime
import time
import os

# Coordinates and Parameters
LATITUDE = -7.0520702239386175
LONGITUDE = 110.43532807750137
TIMEZONE = "Asia/Jakarta"
API_URL = "https://archive-api.open-meteo.com/v1/archive"

# Rate Limiting Config
MAX_RETRIES = 5
BASE_DELAY = 2  # Base delay between requests (seconds)
RETRY_DELAY = 30  # Initial retry delay for 429 errors (seconds)

# Hourly feature list only (no daily)
HOURLY_FEATURES = [
    "temperature_2m", "weather_code", "relative_humidity_2m", "pressure_msl",
    "wind_speed_10m", "rain", "apparent_temperature", "surface_pressure",
    "dew_point_2m", "wind_direction_10m", "wind_gusts_10m"
]

print(f"Hourly features: {len(HOURLY_FEATURES)}")


Hourly features: 11


In [12]:
# Weather Condition Mapping (Optional helper)
def map_weather_code(code):
    '''Maps WMO weather code to a simple condition string.'''
    if code is None:
        return "Unknown"
    if code == 0:
        return "Clear"
    elif code in [1, 2]:
        return "Partially cloudy"
    elif code in [3, 45, 48]:
        return "Overcast"
    elif code in [51, 53, 55]:
        return "Rain"
    elif code in [61, 63, 65]:
        return "Rain, Overcast"
    elif code in [80, 81, 82]:
        return "Rain, Partially cloudy"
    elif code in [95, 96, 99]:
        return "Rain"
    else:
        return "Unknown"


In [13]:
def fetch_hourly_data_chunk(start_date, end_date, retries=0):
    '''Fetch hourly data for a specific date range with retry logic.'''
    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": HOURLY_FEATURES,
        "timezone": TIMEZONE,
    }

    try:
        response = requests.get(API_URL, params=params)

        # Handle rate limiting (429)
        if response.status_code == 429:
            if retries < MAX_RETRIES:
                wait_time = RETRY_DELAY * (2 ** retries)  # Exponential backoff
                print(f"   ⚠️ Rate limited! Waiting {wait_time}s before retry {retries + 1}/{MAX_RETRIES}...")
                time.sleep(wait_time)
                return fetch_hourly_data_chunk(start_date, end_date, retries + 1)
            print(f"   ❌ Max retries reached for {start_date} to {end_date}")
            return None

        response.raise_for_status()
        return response.json()

    except requests.exceptions.RequestException as e:
        if retries < MAX_RETRIES:
            wait_time = RETRY_DELAY * (2 ** retries)
            print(f"   ⚠️ Error: {e}. Retrying in {wait_time}s...")
            time.sleep(wait_time)
            return fetch_hourly_data_chunk(start_date, end_date, retries + 1)
        print(f"   ❌ Failed after {MAX_RETRIES} retries: {e}")
        return None


In [14]:
def fetch_historical_hourly_data():
    '''Fetch hourly data from 2000 to today in yearly chunks.'''
    today = datetime.now()
    start_year = 2000
    end_year = today.year

    all_data = []
    total_years = end_year - start_year + 1

    for idx, year in enumerate(range(start_year, end_year + 1)):
        start_date = f"{year}-01-01"
        end_date = today.strftime("%Y-%m-%d") if year == end_year else f"{year}-12-31"

        print(f"[{idx + 1}/{total_years}] Fetching data for {year} ({start_date} to {end_date})...")

        data = fetch_hourly_data_chunk(start_date, end_date)
        if data is None:
            print(f"   ⏭️ Skipping year {year} due to errors")
            continue

        hourly_data = data.get("hourly", {})
        if not hourly_data:
            print(f"   ⚠️ No hourly data found for {year}.")
            continue

        df_hourly = pd.DataFrame({
            "timestamp": hourly_data.get("time"),
            "temp": hourly_data.get("temperature_2m"),
            "humidity": hourly_data.get("relative_humidity_2m"),
            "windspeed": hourly_data.get("wind_speed_10m"),
            "sealevelpressure": hourly_data.get("pressure_msl"),
            "weather_code": hourly_data.get("weather_code"),
            "rain": hourly_data.get("rain"),
            "apparent_temperature": hourly_data.get("apparent_temperature"),
            "surface_pressure": hourly_data.get("surface_pressure"),
            "dew_point_2m": hourly_data.get("dew_point_2m"),
            "wind_direction_10m": hourly_data.get("wind_direction_10m"),
            "wind_gusts_10m": hourly_data.get("wind_gusts_10m"),
        })

        df_hourly["timestamp"] = pd.to_datetime(df_hourly["timestamp"])
        df_hourly["hour"] = df_hourly["timestamp"].dt.hour
        df_hourly["day"] = df_hourly["timestamp"].dt.day
        df_hourly["month"] = df_hourly["timestamp"].dt.month
        df_hourly["year"] = df_hourly["timestamp"].dt.year
        df_hourly["conditions"] = df_hourly["weather_code"].apply(map_weather_code)

        all_data.append(df_hourly)
        print(f"   ✅ {len(df_hourly):,} records fetched")

        if idx < total_years - 1:  # Don't wait after last request
            print(f"   ⏳ Waiting {BASE_DELAY}s before next request...")
            time.sleep(BASE_DELAY)

    if not all_data:
        print("❌ No data fetched.")
        return None

    df = pd.concat(all_data, ignore_index=True)
    df["id"] = range(len(df))

    output_columns = [
        "id", "timestamp", "hour", "day", "month", "year",
        "temp", "humidity", "windspeed", "sealevelpressure", "rain",
        "apparent_temperature", "surface_pressure", "dew_point_2m",
        "wind_direction_10m", "wind_gusts_10m",
        "weather_code", "conditions",
    ]

    return df[output_columns]


In [16]:
# Execute the data fetch
df = fetch_historical_hourly_data()

if df is not None:
    # Ensure output directory exists
    output_dir = "../data"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    output_file = os.path.join(output_dir, "historical_data_hourly.csv")
    df.to_csv(output_file, index=False)
    print(f"🎉 Data successfully saved to {output_file}")
    print(f"📊 Total records: {len(df):,}")
    print(f"📋 Total columns: {len(df.columns)}")


[1/26] Fetching data for 2000 (2000-01-01 to 2000-12-31)...
   ✅ 8,784 records fetched
   ⏳ Waiting 2s before next request...
[2/26] Fetching data for 2001 (2001-01-01 to 2001-12-31)...
   ✅ 8,760 records fetched
   ⏳ Waiting 2s before next request...
[3/26] Fetching data for 2002 (2002-01-01 to 2002-12-31)...
   ✅ 8,760 records fetched
   ⏳ Waiting 2s before next request...
[4/26] Fetching data for 2003 (2003-01-01 to 2003-12-31)...
   ✅ 8,760 records fetched
   ⏳ Waiting 2s before next request...
[5/26] Fetching data for 2004 (2004-01-01 to 2004-12-31)...
   ✅ 8,784 records fetched
   ⏳ Waiting 2s before next request...
[6/26] Fetching data for 2005 (2005-01-01 to 2005-12-31)...
   ✅ 8,760 records fetched
   ⏳ Waiting 2s before next request...
[7/26] Fetching data for 2006 (2006-01-01 to 2006-12-31)...
   ✅ 8,760 records fetched
   ⏳ Waiting 2s before next request...
[8/26] Fetching data for 2007 (2007-01-01 to 2007-12-31)...
   ⚠️ Error: HTTPSConnectionPool(host='archive-api.open-me

In [17]:
# Preview the data
if df is not None:
    print("=== Column List ===")
    for i, col in enumerate(df.columns):
        print(f"{i+1:2}. {col}")

    print("=== Data Preview (First 5 rows) ===")
    display(df.head())

    print("=== Data Info ===")
    df.info()


=== Column List ===
 1. id
 2. timestamp
 3. hour
 4. day
 5. month
 6. year
 7. temp
 8. humidity
 9. windspeed
10. sealevelpressure
11. rain
12. apparent_temperature
13. surface_pressure
14. dew_point_2m
15. wind_direction_10m
16. wind_gusts_10m
17. weather_code
18. conditions
=== Data Preview (First 5 rows) ===


,id,timestamp,hour,day,month,year,temp,humidity,windspeed,sealevelpressure,rain,apparent_temperature,surface_pressure,dew_point_2m,wind_direction_10m,wind_gusts_10m,weather_code,conditions
0,0,2000-01-01 00:00:00,0,1,1,2000,21.8,98,4.0,1008.4,0.0,25.9,984.5,21.6,270,6.8,3,Overcast
1,1,2000-01-01 01:00:00,1,1,1,2000,21.4,99,4.0,1007.9,0.0,25.3,983.9,21.3,275,9.0,3,Overcast
2,2,2000-01-01 02:00:00,2,1,1,2000,21.4,98,3.2,1007.4,0.0,25.4,983.4,21.2,270,6.8,3,Overcast
3,3,2000-01-01 03:00:00,3,1,1,2000,21.2,99,4.6,1007.0,0.0,24.9,983.0,21.0,252,9.4,3,Overcast
4,4,2000-01-01 04:00:00,4,1,1,2000,21.0,99,3.6,1006.9,0.0,24.7,982.9,20.8,270,10.1,3,Overcast


=== Data Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 227448 entries, 0 to 227447
Data columns (total 18 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   id                    227448 non-null  int64         
 1   timestamp             227448 non-null  datetime64[ns]
 2   hour                  227448 non-null  int32         
 3   day                   227448 non-null  int32         
 4   month                 227448 non-null  int32         
 5   year                  227448 non-null  int32         
 6   temp                  227448 non-null  float64       
 7   humidity              227448 non-null  int64         
 8   windspeed             227448 non-null  float64       
 9   sealevelpressure      227448 non-null  float64       
 10  rain                  227448 non-null  float64       
 11  apparent_temperature  227448 non-null  float64       
 12  surface_pressure      227448 non-null  f

In [18]:
# Summary Statistics
if df is not None:
    print("=== Summary Statistics ===")
    display(df.describe())

    print("=== Missing Values ===")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({
        "Missing Count": missing,
        "Missing %": missing_pct
    })
    display(missing_df[missing_df["Missing Count"] > 0])


=== Summary Statistics ===


,id,timestamp,hour,day,month,year,temp,humidity,windspeed,sealevelpressure,rain,apparent_temperature,surface_pressure,dew_point_2m,wind_direction_10m,wind_gusts_10m,weather_code
count,227448.000000,227448,227448.000000,227448.000000,227448.000000,227448.000000,227448.000000,227448.000000,227448.000000,227448.000000,227448.000000,227448.000000,227448.000000,227448.000000,227448.000000,227448.000000,227448.000000
mean,113723.500000,2012-12-21 11:30:00,11.500000,15.718160,6.511132,2012.473251,25.537303,79.357264,6.157342,1010.092249,0.271224,29.416021,986.395358,21.218579,194.556879,16.848214,17.324386
min,0.000000,2000-01-01 00:00:00,0.000000,1.000000,1.000000,2000.000000,17.000000,16.000000,0.000000,1002.200000,0.000000,17.200000,978.700000,4.400000,1.000000,0.700000,0.000000
25%,56861.750000,2006-06-27 05:45:00,5.750000,8.000000,4.000000,2006.000000,23.200000,71.000000,3.400000,1008.800000,0.000000,27.100000,985.200000,20.200000,133.000000,9.700000,2.000000
50%,113723.500000,2012-12-21 11:30:00,11.500000,16.000000,7.000000,2012.000000,25.000000,83.000000,5.200000,1010.200000,0.000000,29.200000,986.400000,21.900000,171.000000,15.500000,3.000000
75%,170585.250000,2019-06-17 17:15:00,17.250000,23.000000,10.000000,2019.000000,27.500000,93.000000,8.000000,1011.400000,0.100000,31.500000,987.700000,22.800000,286.000000,22.300000,51.000000
max,227447.000000,2025-12-11 23:00:00,23.000000,31.000000,12.000000,2025.000000,37.800000,100.000000,34.700000,1016.800000,33.400000,40.500000,993.000000,27.000000,360.000000,73.800000,65.000000
std,65658.726351,NaN,6.922202,8.801706,3.443212,7.486281,3.021219,16.926806,3.900473,1.866807,0.904223,3.213148,1.793415,2.281746,89.250276,8.937711,23.760495


=== Missing Values ===


,Missing Count,Missing %
